# CTDenoiser — W&B Sweep

**Prerequisites:** Run `00_build_cache.ipynb` once to build `ldct_cache.h5` on your Drive.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

REPO_URL  = 'https://github.com/tsereda/CTDenoiser.git'
BRANCH    = 'main'
REPO_ROOT = '/content/CTDenoiser'

if not os.path.isdir(os.path.join(REPO_ROOT, '.git')):
    !git clone --branch {BRANCH} {REPO_URL} {REPO_ROOT}
else:
    !git -C {REPO_ROOT} pull origin {BRANCH}

%cd {REPO_ROOT}
!pip install -q -e . wandb

In [ ]:
import shutil, os

DRIVE_CACHE = '/content/drive/MyDrive/ldct_cache.h5'
LOCAL_CACHE = '/content/ldct_cache.h5'

if not os.path.exists(LOCAL_CACHE) and os.path.exists(DRIVE_CACHE):
    print('Copying cache to local disk...')
    shutil.copy(DRIVE_CACHE, LOCAL_CACHE)

assert os.path.exists(LOCAL_CACHE), f'Cache not found at {LOCAL_CACHE}'

In [ ]:
import wandb
wandb.login()

## Sweep

In [ ]:
import wandb

WANDB_PROJECT = 'ctdenoiser'
SWEEP_COUNT   = 20

sweep_config = {
    'method': 'bayes',
    'metric': {'name': 'val/psnr', 'goal': 'maximize'},
    'parameters': {
        'model':      {'values': ['redcnn', 'dncnn', 'unet', 'ctformer', 'flowmatching']},
        'lr':         {'values': [1e-5, 1e-4, 1e-3]},
        'batch_size': {'values': [8, 16, 32]},
        'patch_size': {'values': [64, 96]},
        'epochs':     {'value': 1},
    },
}

sweep_id = wandb.sweep(sweep_config, project=WANDB_PROJECT)
print('Sweep ID:', sweep_id)

In [ ]:
import subprocess, sys, os
import wandb

def train_run():
    run = wandb.init()
    cfg = run.config
    run_id = run.id
    # Finish parent run before subprocess opens the same run ID.
    run.finish()

    cmd = [
        sys.executable, '-m', 'ctdenoiser.train',
        '--model',           cfg.model,
        '--lr',              str(cfg.lr),
        '--batch-size',      str(cfg.batch_size),
        '--patch-size',      str(cfg.patch_size),
        '--epochs',          str(cfg.epochs),
        '--h5-cache',        LOCAL_CACHE,
        '--wandb-project',   WANDB_PROJECT,
        '--flow-steps-eval', '5',
    ]
    if cfg.model == 'flowmatching':
        cmd += ['--flow-steps', '20']

    env = {**os.environ, 'WANDB_RUN_ID': run_id, 'WANDB_RESUME': 'must'}
    subprocess.run(cmd, env=env, check=True, cwd=REPO_ROOT)


wandb.agent(sweep_id, function=train_run, count=SWEEP_COUNT)